# 13. Slope and walking time

This notebook asks whether the results on walking access to food change when
the walk is timed over the terrain rather than over the map. The main measure
converts network distance to time at one speed on every street. Many of the 92
metropolitan areas are hilly, and a resident who walks uphill to a shop takes
longer than the distance suggests.

Elevation comes from a digital elevation model at 15 m resolution, one raster
for each metropolitan area. Every node of the pedestrian network receives an
elevation, every edge receives a grade, and the time to walk each edge in each
direction follows the hiking function of Tobler (1993), which gives the speed
of a walker as a function of grade and is the standard choice for slope-aware
walking times. The function is scaled so that its speed on flat ground equals
the 4.8 km/h of the main measure, so the only difference between the two
measures is the terrain.

For the two anchor definitions, supermarkets and warehouse clubs and the
specialist fresh and staple sellers, the notebook measures the time from every
inhabited block to the nearest destination, both for the walk to the outlet and
for the walk back, and expresses each time as the length of a flat walk that
takes as long. That slope adjusted distance is then summarised by census tract
and the model of the deprivation gradient is fitted to it on the estimation
sample, so that its coefficient can be read beside the baseline.

Outputs:

| file | content |
| --- | --- |
| `Results/13_block_slope.parquet` | one row per inhabited block: minutes and slope adjusted distance to each anchor, to the outlet and back |
| `Results/13_area_slope.parquet` | one row per area: terrain of the network and the effect of slope on the median walk |
| `Results/13_slope_gradient.parquet` | the gradient model fitted to the slope adjusted distances, read by notebook 12 for the robustness table |

The run writes one consolidated report, offered for download at the end.

## 1. Setup

In [ ]:
import sys, os, glob, zipfile, hashlib, importlib, time, math, gc

if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

PROJECT = os.environ.get('MXO_BASE', '/content/drive/MyDrive/Mexico_Obesity')
os.environ['MXO_BASE'] = PROJECT
sys.path.insert(0, f'{PROJECT}/Code')
import mxo_utils as mx
importlib.reload(mx)
assert mx.BASE == PROJECT, f'the module resolves to {mx.BASE}'

import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
import pyogrio
from pyproj import Transformer
from scipy import stats
from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix, csr_matrix
from scipy.sparse.csgraph import dijkstra
from scipy.ndimage import map_coordinates

try:
    import rasterio
    import rasterio.windows
except ImportError:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rasterio'],
                   check=True)
    import rasterio
    import rasterio.windows

rep = mx.Report('13_slope_robustness').start()

INPUTS = mx.DIRS['inputs']
RESULTS = mx.DIRS['results']
DEM_DIR = os.path.join(INPUTS, 'DEM')

WALK_KMH = 4.8
WALK_M_PER_MIN = WALK_KMH * 1000 / 60
TOBLER_FLAT_KMH = 6.0 * math.exp(-3.5 * 0.05)   # Tobler's speed at zero grade
DENSIFY_M = 50.0         # longest piece a way is divided into, as in notebook 07
NODE_PRECISION = 2
MIN_EDGE_M = 1e-6
GRADE_CAP = 0.35         # grades beyond this are held at this value, see section 3
ANCHORS = ['supermarket_club', 'fresh_only']
LABEL = {'supermarket_club': 'Supermarkets and clubs',
         'fresh_only': 'Fresh and staple specialists'}

print(f'flat walking speed {WALK_KMH} km/h, {WALK_M_PER_MIN:.1f} metres a minute')
print(f"Tobler's function gives {TOBLER_FLAT_KMH:.3f} km/h on the flat and is "
      f'scaled by {WALK_KMH / TOBLER_FLAT_KMH:.4f} to match')

## 2. Networks, blocks, destinations and elevation rasters

The frame, the destinations and the networks are read exactly as in notebook
07, and each elevation raster is assigned to the area whose name appears in
its file name, in the same way as the network files. The raster projection is
read from the file rather than assumed, and node coordinates are transformed
into it before elevation is sampled.

In [ ]:
BLOCKS = pd.read_parquet(os.path.join(RESULTS, '04_blocks.parquet'),
                         columns=['CVEGEO_MZA', 'CVEGEO', 'slug',
                                  'macro_region', 'POBTOT'])
BLOCKS['inhabited'] = BLOCKS['POBTOT'] > 0
OUT = pd.read_parquet(os.path.join(RESULTS, '05_outlets.parquet'),
                      columns=['slug', 'x_m', 'y_m'] + ANCHORS)
METRO = gpd.read_file(os.path.join(RESULTS, '03_metro_boundaries.gpkg'),
                      layer='metro_boundaries')
AREAS = sorted(METRO['slug'], key=len, reverse=True)


def assign_files(files, what):
    """One file per area, matched by the longest area name in the file name."""
    stems = {f: mx._slug_name(
        [os.path.splitext(os.path.basename(f))[0]]).iloc[0] for f in files}
    found, orphan, clash = {}, [], []
    for f in files:
        hit = next((a for a in AREAS if a in stems[f]), None)
        if hit is None:
            orphan.append(f)
        elif hit in found:
            clash.append((hit, found[hit], f))
        else:
            found[hit] = f
    absent = sorted(set(METRO['slug']) - set(found))
    if absent or orphan or clash:
        print(f'{len(files)} {what} files\n')
        for f in files:
            print(f'   {os.path.basename(f):60s} -> {stems[f]}')
        print(f'\nareas with no file ({len(absent)}): {absent}')
        print(f'files matching no area ({len(orphan)}): '
              f'{[os.path.basename(f) for f in orphan]}')
        print(f'areas matched by more than one file ({len(clash)}): '
              f'{[(a, os.path.basename(x), os.path.basename(y)) for a, x, y in clash]}')
        raise RuntimeError(f'the {what} files do not correspond one to one '
                           'with the 92 areas; the listing above shows which')
    return found


NETDIR = next(d for d in mx.NETWORKS_CANDIDATES if os.path.isdir(d))
NETS = assign_files(sorted(glob.glob(os.path.join(NETDIR, '*.gpkg'))), 'network')
dem_files = sorted(f for f in glob.glob(os.path.join(DEM_DIR, '*'))
                   if f.lower().endswith(('.tif', '.tiff')))
assert dem_files, f'no .tif files in {DEM_DIR}'
DEMS = assign_files(dem_files, 'elevation')

with rasterio.open(DEMS[AREAS[0]]) as src:
    print(f'elevation rasters in {DEM_DIR}')
    print(f'first raster {os.path.basename(DEMS[AREAS[0]])}: crs {src.crs}, '
          f'{src.width} by {src.height} cells, nodata {src.nodata}')

print(f'{len(BLOCKS):,} blocks, {int(BLOCKS.inhabited.sum()):,} inhabited')
print(f'{len(OUT):,} located establishments')
print(f'{len(NETS)} networks and {len(DEMS)} rasters, one of each for each area')
rep.add_table('network and raster for each area', pd.DataFrame(
    {'slug': list(NETS), 'network': [os.path.basename(NETS[s]) for s in NETS],
     'raster': [os.path.basename(DEMS[s]) for s in NETS]}), max_rows=92)

## 3. Block geometry and the graph

The block points and the graph are built as in notebook 07: every way is
divided into pieces of at most fifty metres, every vertex becomes a node and
every pair of consecutive vertices an edge with its planar length.

Each node is then given the elevation of the raster at its position, read by
bilinear interpolation between the four nearest cells, so that a node between
cells does not jump to the value of one of them. The grade of an edge is the
elevation difference between its ends divided by its planar length. Because the
raster has a step of 15 m and some edges are shorter than that, a small error in
elevation can produce an implausible grade on a short edge, and a bridge,
underpass or retaining wall produces one on any edge. Grades are therefore held
at 0.35 in absolute value. At that grade Tobler's function gives a speed under
1.5 km/h, slower than a staircase, so a street steeper than that is either not
walked as a street or not a street at all. The share of edges affected is
reported for every area.

Tobler's hiking function gives the speed on an edge of grade $S$ as
$6\exp(-3.5\,|S + 0.05|)$ km/h, fastest on a slight descent and slower both
uphill and downhill from there. The time of an edge therefore differs by
direction. The shortest path search runs outward from the destinations, so each
edge is entered in the search with the time of walking it towards the
destination, and the search returns the time of the walk from every block to
its nearest outlet. The walk back is found by entering the opposite times. A
block's attachment and a destination's attachment are counted at the flat speed.

In [ ]:
LOCAL = '/content/mgn_blocks'
os.makedirs(LOCAL, exist_ok=True)


def locate_block_layer():
    """The archive member holding block polygons keyed by a 16 character key."""
    for arch in sorted(glob.glob(os.path.join(INPUTS, 'MGN', '**', '*.zip'),
                                 recursive=True)):
        with zipfile.ZipFile(arch) as zf:
            shps = [n for n in zf.namelist() if n.lower().endswith('.shp')]
        for shp in shps:
            try:
                info = pyogrio.read_info(f'/vsizip/{arch}/{shp}')
            except Exception:
                continue
            if 'CVEGEO' not in list(info['fields']):
                continue
            if 'Polygon' not in str(info['geometry_type']):
                continue
            head = pyogrio.read_dataframe(f'/vsizip/{arch}/{shp}',
                                          columns=['CVEGEO'], max_features=5,
                                          read_geometry=False)
            if head['CVEGEO'].astype(str).str.len().eq(16).all():
                return arch, shp, info
    raise RuntimeError('no block layer found under Inputs/MGN')


ARCH, SHP, INFO = locate_block_layer()
stem = os.path.splitext(os.path.basename(SHP))[0]
BLOCK_PATH = os.path.join(LOCAL, stem + '.shp')
if not os.path.exists(BLOCK_PATH):
    with mx.stage('extract block layer'):
        with zipfile.ZipFile(ARCH) as zf:
            for n in zf.namelist():
                if os.path.splitext(n)[0] == os.path.splitext(SHP)[0]:
                    zf.extract(n, LOCAL)
                    src = os.path.join(LOCAL, n)
                    dst = os.path.join(LOCAL, os.path.basename(n))
                    if src != dst:
                        os.replace(src, dst)
BLOCK_CRS = pyogrio.read_info(BLOCK_PATH)['crs']
print('block layer:', os.path.basename(SHP), '| crs', BLOCK_CRS)


def block_points(slug, boundary, keys):
    bounds = (gpd.GeoSeries([boundary], crs=METRO.crs)
                .to_crs(BLOCK_CRS).total_bounds)
    G = pyogrio.read_dataframe(BLOCK_PATH, columns=['CVEGEO'], bbox=tuple(bounds))
    G = G[G['CVEGEO'].isin(keys)]
    G = G[G.geometry.notna() & ~G.geometry.is_empty].drop_duplicates('CVEGEO')
    return gpd.GeoDataFrame({'CVEGEO_MZA': G['CVEGEO'].values},
                            geometry=G.geometry.representative_point().values,
                            crs=G.crs)


def build_edges(path):
    """Node coordinates and the edge list of a network, divided at DENSIFY_M."""
    layers = [name for name, kind in pyogrio.list_layers(path)
              if 'Line' in str(kind)]
    name = 'edges' if 'edges' in layers else layers[0]
    E = pyogrio.read_dataframe(path, layer=name, columns=[])
    E = E[E.geometry.notna() & ~E.geometry.is_empty]
    E = E.explode(index_parts=False, ignore_index=True)
    E = E[(E.geometry.geom_type == 'LineString') & (E.geometry.length > 0)]
    geom = shapely.segmentize(E.geometry.values, DENSIFY_M)
    pts, which = shapely.get_coordinates(geom, return_index=True)
    same = which[:-1] == which[1:]
    p0, p1 = pts[:-1][same], pts[1:][same]
    seg = np.hypot(p1[:, 0] - p0[:, 0], p1[:, 1] - p0[:, 1])
    keep = seg > 0
    p0, p1, seg = p0[keep], p1[keep], seg[keep]
    stacked = np.round(np.vstack([p0, p1]), NODE_PRECISION)
    coords, inverse = np.unique(stacked, axis=0, return_inverse=True)
    inverse = np.asarray(inverse).ravel()
    half = len(p0)
    u, v = inverse[:half], inverse[half:]
    ed = pd.DataFrame({'u': np.minimum(u, v), 'v': np.maximum(u, v),
                       'w': np.maximum(seg, MIN_EDGE_M)})
    ed = ed[ed.u != ed.v].groupby(['u', 'v'], sort=False)['w'].min().reset_index()
    return coords, ed, E.crs


def sample_elevation(path, coords, net_crs, tile=4096):
    """Elevation at each node by bilinear interpolation, read tile by tile.

    The raster is never held whole in memory. Nodes are grouped by the tile of
    the raster they fall in, each tile is read with a margin of two cells as
    32-bit floats, and the nodes in it are interpolated. A node whose four
    surrounding cells include a missing value receives no elevation here and is
    given that of the nearest node with one afterwards.
    """
    z = np.full(len(coords), np.nan)
    with rasterio.open(path) as src:
        tf = Transformer.from_crs(net_crs, src.crs, always_xy=True)
        xs, ys = tf.transform(coords[:, 0], coords[:, 1])
        inv = ~src.transform
        cols, rows = inv * (np.asarray(xs), np.asarray(ys))
        cols, rows = np.asarray(cols) - 0.5, np.asarray(rows) - 0.5
        inside = ((rows >= 0) & (rows <= src.height - 1)
                  & (cols >= 0) & (cols <= src.width - 1))
        idx = np.flatnonzero(inside)
        if not len(idx):
            return z
        tr_, tc_ = (rows[idx] // tile).astype(int), (cols[idx] // tile).astype(int)
        key = tr_ * 100000 + tc_
        for k in np.unique(key):
            sel = idx[key == k]
            r0 = max(int(np.floor(rows[sel].min())) - 2, 0)
            c0 = max(int(np.floor(cols[sel].min())) - 2, 0)
            r1 = min(int(np.ceil(rows[sel].max())) + 3, src.height)
            c1 = min(int(np.ceil(cols[sel].max())) + 3, src.width)
            win = rasterio.windows.Window(c0, r0, c1 - c0, r1 - r0)
            arr = src.read(1, window=win, out_dtype='float32')
            if src.nodata is not None:
                arr[arr == src.nodata] = np.nan
            z[sel] = map_coordinates(arr, [rows[sel] - r0, cols[sel] - c0],
                                     order=1, mode='nearest', cval=np.nan)
            del arr
    return z


def tobler_minutes(length_m, grade):
    """Minutes to walk an edge of the given length at the given grade, with
    Tobler's function scaled to WALK_KMH on the flat."""
    kmh = 6.0 * np.exp(-3.5 * np.abs(grade + 0.05)) * (WALK_KMH / TOBLER_FLAT_KMH)
    return length_m / 1000.0 / kmh * 60.0


def terrain_edges(slug):
    """Node coordinates, the edge list with lengths and grades, and the terrain
    summary of one network."""
    coords, ed, net_crs = build_edges(NETS[slug])
    n = len(coords)
    z = sample_elevation(DEMS[slug], coords, net_crs)
    missing = np.isnan(z)
    if missing.any() and not missing.all():
        tree = cKDTree(coords[~missing])
        _, j = tree.query(coords[missing], workers=-1)
        z[missing] = z[~missing][j]
        del tree
    elif missing.all():
        z[:] = 0.0
    u = ed.u.to_numpy(np.int64)
    v = ed.v.to_numpy(np.int64)
    L = ed.w.to_numpy(float)
    del ed
    raw = (z[v] - z[u]) / L
    grade = np.clip(raw, -GRADE_CAP, GRADE_CAP)
    terrain = {'nodes': n, 'edges': len(L), 'network_km': round(L.sum() / 1000, 1),
               'nodes_without_elevation': int(missing.sum()),
               'share_nodes_without_elevation': round(float(missing.mean()), 5),
               'elevation_min_m': round(float(np.nanmin(z)), 1),
               'elevation_max_m': round(float(np.nanmax(z)), 1),
               'mean_abs_grade': round(float(np.average(np.abs(raw), weights=L)), 4),
               'share_length_grade_over_5pct': round(
                   float(L[np.abs(raw) > 0.05].sum() / L.sum()), 4),
               'share_length_grade_over_10pct': round(
                   float(L[np.abs(raw) > 0.10].sum() / L.sum()), 4),
               'share_edges_capped': round(float((np.abs(raw) > GRADE_CAP).mean()), 5)}
    del raw, z
    return coords, net_crs, u, v, L, grade, terrain


def adjacency(n, u, v, w_uv, w_vu):
    """Directed adjacency with one spare row and column for the auxiliary source.

    Entry (a, b) carries the cost of moving from a to b in the search. The
    search runs outward from the destinations, so the cost stored from a to b
    must be the cost of walking from b to a for the result to be the walk
    towards the destination.
    """
    return coo_matrix((np.concatenate([w_vu, w_uv]),
                       (np.concatenate([u, v]), np.concatenate([v, u]))),
                      shape=(n + 1, n + 1)).tocsr()


def nearest_with_offsets(W, n, dest_nodes, dest_offsets):
    order = np.argsort(dest_offsets, kind='stable')
    nodes, first = np.unique(dest_nodes[order], return_index=True)
    offsets = np.maximum(dest_offsets[order][first], MIN_EDGE_M)
    aux = csr_matrix((offsets, (np.full(len(nodes), n), nodes)),
                     shape=(n + 1, n + 1))
    d = dijkstra(W + aux, directed=True, indices=n)
    return d[:n]

## 4. Every metropolitan area

For each area and each anchor definition, three searches are run: distance, as
in the main measure, time of the walk to the outlet, and time of the walk back.
Each time is also expressed as the slope adjusted distance, the length of a flat
walk that takes as long. The distance search repeats the main measure and is
kept only to confirm that this notebook reproduces it.

The largest networks hold several million nodes, so memory is managed
explicitly. The elevation raster is read in tiles around the nodes rather than
whole, only one of the three adjacencies is held at a time, and each area runs
in its own child process, which returns all of its memory to the system when it
ends. The peak memory of each area is printed. The loop also stops after a set
number of minutes; running the cell again continues from the areas already
measured.

In [ ]:
import hashlib, time, os, gc, resource
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
from concurrent.futures.process import BrokenProcessPool

TIME_BUDGET_MIN = 150    # the loop stops cleanly after this many minutes and resumes on the next run


def dem_fingerprint():
    h = hashlib.blake2b(digest_size=4)
    for s in sorted(DEMS):
        st = os.stat(DEMS[s])
        h.update(f'{s}|{st.st_size}|'.encode())
    return h.hexdigest()


KEY = hashlib.blake2b(('|'.join(ANCHORS) + f'|{WALK_KMH}|{NODE_PRECISION}|'
                       f'{DENSIFY_M}|{GRADE_CAP}|tobler|' + dem_fingerprint())
                      .encode(), digest_size=4).hexdigest()
CACHE = mx.cache_dir(f'cache_13_slope_{KEY}')
print('cache directory:', CACHE)


def measure(slug, boundary):
    """Distance and walking times from each inhabited block of one area to
    each anchor set. One adjacency is held in memory at a time."""
    frame = BLOCKS[BLOCKS.slug == slug]
    coords, net_crs, u, v, L, grade, terrain = terrain_edges(slug)
    n = len(coords)
    tree = cKDTree(coords)
    pts = block_points(slug, boundary, set(frame.CVEGEO_MZA)).to_crs(net_crs)
    pts = pts.merge(frame[['CVEGEO_MZA', 'POBTOT', 'inhabited']], on='CVEGEO_MZA')
    bxy = np.column_stack([pts.geometry.x.values, pts.geometry.y.values])
    bsnap, bnode = tree.query(bxy, workers=-1)
    dest = OUT[OUT.slug == slug]
    dgeo = gpd.GeoSeries(gpd.points_from_xy(dest.x_m, dest.y_m),
                         crs=f'EPSG:{mx.CRS_METRIC}').to_crs(net_crs)
    dxy = np.column_stack([dgeo.x.values, dgeo.y.values])
    dsnap, dnode = tree.query(dxy, workers=-1)
    del tree, coords, dgeo

    out = {'CVEGEO_MZA': pts.CVEGEO_MZA.values, 'POBTOT': pts.POBTOT.values}
    t_walk = None
    # (name, cost of walking from u to v, cost of walking from v to u, unit)
    plans = [('distance_m', lambda: (L, L), 1.0),
             ('minutes_to', lambda: (tobler_minutes(L, grade),
                                     tobler_minutes(L, -grade)), WALK_M_PER_MIN),
             ('minutes_back', lambda: (tobler_minutes(L, -grade),
                                       tobler_minutes(L, grade)), WALK_M_PER_MIN)]
    for name, costs, unit in plans:
        c_uv, c_vu = costs()
        W = adjacency(n, u, v, c_uv, c_vu)
        del c_uv, c_vu
        for s in ANCHORS:
            m = dest[s].values
            if not m.any():
                out[f'{name}_{s}'] = np.full(len(pts), np.nan)
                continue
            d = nearest_with_offsets(W, n, dnode[m], dsnap[m] / unit)
            val = d[bnode] + bsnap / unit
            val[np.isinf(d[bnode])] = np.nan
            out[f'{name}_{s}'] = val
            del d
        del W
        gc.collect()

    R = pd.DataFrame(out)
    R['slug'] = slug
    R = R[pts.inhabited.values].reset_index(drop=True)
    meta = {'slug': slug, 'blocks': int(pts.inhabited.sum()), **terrain}
    return R, meta


def run_area(slug):
    """Measure one area and write it to the cache. Runs in a child process,
    so that all of its memory is returned when it ends."""
    fp = os.path.join(CACHE, f'{slug}.parquet')
    mp = os.path.join(CACHE, f'{slug}_meta.parquet')
    t = time.time()
    R, meta = measure(slug, GEOM.loc[slug])
    meta['seconds'] = round(time.time() - t, 1)
    meta['peak_rss_gb'] = round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6, 2)
    R.to_parquet(fp, index=False)
    pd.DataFrame([meta]).to_parquet(mp, index=False)
    return slug, meta['seconds'], meta['peak_rss_gb']


order = (BLOCKS.groupby('slug')['POBTOT'].sum()
               .sort_values(ascending=False).index.tolist())
GEOM = METRO.set_index('slug').geometry


def cached(slug):
    return (os.path.exists(os.path.join(CACHE, f'{slug}.parquet'))
            and os.path.exists(os.path.join(CACHE, f'{slug}_meta.parquet')))


# Each area runs in its own child process, started by fork so that it shares
# the tables already read, and ended after the area so that the memory of the
# graph is returned in full. If the child is stopped for lack of memory, the
# area is reported and the loop moves on. Areas run smallest first, so that a
# failure in one of the largest does not hold back the rest.
todo = [s_ for s_ in reversed(order) if not cached(s_)]
print(f'{len(order) - len(todo)} areas already measured, {len(todo)} to go')
ctx = mp.get_context('fork')
failed, pending = [], []
t0 = time.time()
for i, slug in enumerate(todo, 1):
    if (time.time() - t0) / 60 > TIME_BUDGET_MIN:
        pending = todo[i - 1:]
        break
    with ProcessPoolExecutor(max_workers=1, mp_context=ctx) as ex:
        try:
            _, sec, rss = ex.submit(run_area, slug).result()
            print(f'[{i:>2}/{len(todo)}] {slug:28s} {sec:7.1f} s   peak {rss:5.2f} GB')
        except BrokenProcessPool:
            failed.append(slug)
            print(f'[{i:>2}/{len(todo)}] {slug:28s} stopped, out of memory')
    gc.collect()

if pending:
    print(f'\ntime budget spent after {(time.time() - t0) / 60:.0f} minutes; '
          f'{len(pending)} areas remain: {pending}')
if failed:
    print(f'\nareas stopped for lack of memory: {failed}')
if pending or failed:
    raise RuntimeError('run this cell again to continue; measured areas are read '
                       'from the cache. An area that runs out of memory twice '
                       'needs a runtime with more memory')

parts = [pd.read_parquet(os.path.join(CACHE, f'{s_}.parquet')) for s_ in order]
metas = [pd.read_parquet(os.path.join(CACHE, f'{s_}_meta.parquet')) for s_ in order]
A = pd.concat(parts, ignore_index=True)
A = A.merge(BLOCKS[['CVEGEO_MZA', 'macro_region']], on='CVEGEO_MZA', how='left')
TER = pd.concat(metas, ignore_index=True)
print(f'\n{len(A):,} inhabited blocks measured in {time.time() - t0:.0f}s')

for s in ANCHORS:
    A[f'minutes_flat_{s}'] = A[f'distance_m_{s}'] / WALK_M_PER_MIN
    A[f'minutes_round_{s}'] = 0.5 * (A[f'minutes_to_{s}'] + A[f'minutes_back_{s}'])
    for k in ('to', 'back', 'round'):
        A[f'slope_m_{k}_{s}'] = A[f'minutes_{k}_{s}'] * WALK_M_PER_MIN

## 5. Checks against the main measure

The distance search here must return the distances of notebook 07 for the same
blocks, since the graph, the attachment and the destinations are the same. Any
difference beyond rounding would mean the two notebooks measure different
things.

In [ ]:
import os
import numpy as np
import pandas as pd

MAIN = pd.read_parquet(os.path.join(RESULTS, '07_block_access.parquet'),
                       columns=['CVEGEO_MZA'] + [f'distance_m_{s}' for s in ANCHORS])
C = A[['CVEGEO_MZA'] + [f'distance_m_{s}' for s in ANCHORS]].merge(
    MAIN, on='CVEGEO_MZA', suffixes=('', '_main'))
print(f'blocks in both files: {len(C):,} of {len(A):,}')
assert len(C) == len(A), 'the block frame differs from notebook 07'
for s in ANCHORS:
    a, b = C[f'distance_m_{s}'], C[f'distance_m_{s}_main']
    both = a.notna() & b.notna()
    gap = float((a[both] - b[both]).abs().max())
    print(f'{s:18s} largest difference from notebook 07 {gap:.6f} m; '
          f'present here but not there {int((a.notna() & b.isna()).sum())}, '
          f'there but not here {int((a.isna() & b.notna()).sum())}')
    assert gap < 0.01, s

print(f"\nelevation missing at {int(TER.nodes_without_elevation.sum()):,} of "
      f"{int(TER.nodes.sum()):,} nodes, filled from the nearest node with data")
print(f"edges held at the grade cap: "
      f"{float((TER.share_edges_capped * TER.edges).sum() / TER.edges.sum()):.5f} of all edges")
for s in ANCHORS:
    worse = (A[f'minutes_to_{s}'] < A[f'minutes_flat_{s}'] - 1e-9).sum()
    print(f'{s:18s} blocks whose walk to the outlet is quicker than the flat walk: '
          f'{int(worse):,} (a slight descent is the fastest grade)')

## 6. Terrain and the effect of slope by area

The share of network length steeper than five and ten per cent describes the
terrain, and the ratio of the median slope adjusted walk to the median flat walk
describes what slope does to the access measure in each area.

In [ ]:
def wq(frame, col, q, weight='POBTOT'):
    f = frame[[col, weight]].dropna()
    if not len(f):
        return np.nan
    f = f.sort_values(col)
    cw = f[weight].cumsum() / f[weight].sum()
    return float(f.loc[cw >= q, col].iloc[0])


metros = mx.load_metros()
names = metros.assign(slug=mx._slug_name(metros['metro_key']).values)[
    ['slug', 'metro_name', 'tipo']]
region = BLOCKS.groupby('slug')['macro_region'].agg(lambda v: v.mode().iloc[0])

rows = []
for slug, sub in A.groupby('slug'):
    r = {'slug': slug, 'population': int(sub.POBTOT.sum())}
    for s in ANCHORS:
        flat = wq(sub, f'distance_m_{s}', 0.5)
        to = wq(sub, f'slope_m_to_{s}', 0.5)
        back = wq(sub, f'slope_m_back_{s}', 0.5)
        r[f'median_flat_m_{s}'] = round(flat)
        r[f'median_to_m_{s}'] = round(to)
        r[f'median_back_m_{s}'] = round(back)
        r[f'ratio_to_{s}'] = round(to / flat, 3) if flat else np.nan
        r[f'ratio_back_{s}'] = round(back / flat, 3) if flat else np.nan
    rows.append(r)
AREA = (pd.DataFrame(rows).merge(TER, on='slug').merge(names, on='slug', how='left')
          .assign(macro_region=lambda d: d.slug.map(region)))
print('--- areas with the steepest networks ---\n')
print(AREA.nlargest(12, 'share_length_grade_over_10pct')[
    ['metro_name', 'macro_region', 'mean_abs_grade', 'share_length_grade_over_5pct',
     'share_length_grade_over_10pct', 'ratio_to_supermarket_club', 'ratio_to_fresh_only']]
    .to_string(index=False))
print('\n--- areas where slope lengthens the walk to a supermarket the most ---\n')
print(AREA.nlargest(12, 'ratio_to_supermarket_club')[
    ['metro_name', 'macro_region', 'median_flat_m_supermarket_club',
     'median_to_m_supermarket_club', 'ratio_to_supermarket_club',
     'median_flat_m_fresh_only', 'median_to_m_fresh_only', 'ratio_to_fresh_only']]
    .to_string(index=False))
rep.add_table('terrain and the effect of slope by area', AREA, max_rows=92)

print('\n--- all areas and macro regions, residents ---\n')
rows = []
groups = [('All 92 areas', A)] + [(g, sub) for g, sub in A.groupby('macro_region')]
for g, sub in groups:
    r = {'stratum': g, 'population': int(sub.POBTOT.sum())}
    for s in ANCHORS:
        r[f'flat_m_{s}'] = round(wq(sub, f'distance_m_{s}', 0.5))
        r[f'to_min_{s}'] = round(wq(sub, f'minutes_to_{s}', 0.5), 1)
        r[f'flat_min_{s}'] = round(wq(sub, f'minutes_flat_{s}', 0.5), 1)
        r[f'to_m_{s}'] = round(wq(sub, f'slope_m_to_{s}', 0.5))
        r[f'back_m_{s}'] = round(wq(sub, f'slope_m_back_{s}', 0.5))
        r[f'ratio_to_{s}'] = round(r[f'to_m_{s}'] / r[f'flat_m_{s}'], 3)
    tl = TER.merge(region.rename('macro_region'), left_on='slug', right_index=True)
    tl = tl if g == 'All 92 areas' else tl[tl.macro_region == g]
    km = tl.network_km
    r['share_length_grade_over_5pct'] = round(
        float((tl.share_length_grade_over_5pct * km).sum() / km.sum()), 4)
    r['share_length_grade_over_10pct'] = round(
        float((tl.share_length_grade_over_10pct * km).sum() / km.sum()), 4)
    rows.append(r)
STRATA = pd.DataFrame(rows)
print(STRATA.to_string(index=False))
rep.add_table('slope and the median walk by macro region', STRATA, max_rows=10)

## 7. The deprivation gradient on slope adjusted distance

The model of the main analysis, log of one plus the tract median distance on
the standardised marginalisation index and log population density with area
fixed effects, population weights and errors clustered on areas, is fitted to
the slope adjusted distances on the estimation sample defined in notebook 08.
The baseline is refitted on the same tracts first, so that any difference
between it and the published coefficient would show that the sample differs.

In [ ]:
import os, math
import numpy as np
import pandas as pd
from scipy import stats

TR = pd.read_parquet(os.path.join(RESULTS, '08_tract_access.parquet'),
                     columns=['CVEGEO', 'slug', 'population', 'imu_depriv_z',
                              'pop_density_km2', 'estimation_sample']
                             + [f'median_m_{s}' for s in ANCHORS])
TR = TR[TR['estimation_sample'].fillna(False).astype(bool)].reset_index(drop=True)
A['CVEGEO'] = A['CVEGEO_MZA'].astype(str).str[:13]


def group_wmedian(frame, by, col, weight):
    f = frame[[by, col, weight]].dropna(subset=[col])
    f = f[f[weight] > 0].sort_values([by, col], kind='stable')
    tot = f.groupby(by, sort=False)[weight].transform('sum')
    cum = f.groupby(by, sort=False)[weight].cumsum()
    return f[cum >= 0.5 * tot].groupby(by, sort=False)[col].first()


MEAS = {}
for s in ANCHORS:
    for k in ('to', 'back', 'round'):
        MEAS[f'{k}_{s}'] = group_wmedian(A, 'CVEGEO', f'slope_m_{k}_{s}', 'POBTOT')
    MEAS[f'flat_{s}'] = group_wmedian(A, 'CVEGEO', f'distance_m_{s}', 'POBTOT')
for k, v in MEAS.items():
    TR[k] = v.reindex(TR['CVEGEO']).to_numpy(float)
for s in ANCHORS:
    gap = float((TR[f'flat_{s}'] - TR[f'median_m_{s}']).abs().max())
    print(f'{s:18s} tract medians against notebook 08, largest difference {gap:.4f} m')
    assert gap < 0.5, s

w = TR['population'].to_numpy(float)
xdep = TR['imu_depriv_z'].to_numpy(float)
logden = np.log(TR['pop_density_km2'].to_numpy(float))
gcode = pd.factorize(TR['slug'])[0]
print(f'{len(TR):,} tracts, {w.sum():,.0f} people, {gcode.max() + 1} areas')


def demean(M, w, g):
    M = np.asarray(M, float)
    if M.ndim == 1:
        M = M[:, None]
    sw = np.bincount(g, weights=w)
    out = np.empty_like(M)
    for j in range(M.shape[1]):
        m = np.bincount(g, weights=w * M[:, j]) / sw
        out[:, j] = M[:, j] - m[g]
    return out


def fit(y, X, w, g):
    ok = np.isfinite(y)
    y, X, w, g = y[ok], X[ok], w[ok], pd.factorize(g[ok])[0]
    yd, Xd = demean(y, w, g)[:, 0], demean(X, w, g)
    sw = np.sqrt(w)
    beta = np.linalg.lstsq(Xd * sw[:, None], yd * sw, rcond=None)[0]
    e = yd - Xd @ beta
    bread = np.linalg.inv((Xd * w[:, None]).T @ Xd)
    S = Xd * (w * e)[:, None]
    G = int(g.max()) + 1
    Sg = np.zeros((G, Xd.shape[1]))
    np.add.at(Sg, g, S)
    n, k = Xd.shape
    c = G / (G - 1) * (n - 1) / (n - k)
    V = c * bread @ (Sg.T @ Sg) @ bread
    se = np.sqrt(np.diag(V))
    t = beta / se
    return dict(beta=beta, se=se, t=t, p=2 * stats.norm.sf(np.abs(t)), n=n,
                bread=bread, Sg=Sg, c=c)


def cross_cov(f1, f2):
    C = f1['c'] * f1['bread'] @ (f1['Sg'].T @ f2['Sg']) @ f2['bread']
    return float(C[0, 0])


X = np.column_stack([xdep, logden])
VARIANTS = [('flat', 'Baseline, distance at one speed'),
            ('to', 'Slope adjusted, walk to the outlet'),
            ('back', 'Slope adjusted, walk back from the outlet'),
            ('round', 'Slope adjusted, mean of both directions')]
rows, FITS = [], {}
for k, lab in VARIANTS:
    for s in ANCHORS:
        f = fit(np.log1p(TR[f'{k}_{s}'].to_numpy(float)), X, w, gcode)
        FITS[(k, s)] = f
        rows.append({'variant': k, 'label': lab, 'destination set': s,
                     'coefficient': f['beta'][0], 'standard error': f['se'][0],
                     't': f['t'][0], 'p': f['p'][0],
                     'change per SD, per cent': 100 * (math.exp(f['beta'][0]) - 1),
                     'density coefficient': f['beta'][1], 'tracts': f['n']})
    f1, f2 = FITS[(k, ANCHORS[0])], FITS[(k, ANCHORS[1])]
    diff = f1['beta'][0] - f2['beta'][0]
    sed = math.sqrt(f1['se'][0] ** 2 + f2['se'][0] ** 2 - 2 * cross_cov(f1, f2))
    rows.append({'variant': k, 'label': lab, 'destination set': 'difference',
                 'coefficient': diff, 'standard error': sed, 't': diff / sed,
                 'p': 2 * stats.norm.sf(abs(diff / sed)),
                 'change per SD, per cent': np.nan, 'density coefficient': np.nan,
                 'tracts': f1['n']})
GRAD = pd.DataFrame(rows)
pd.set_option('display.width', 200)
print(GRAD.round(4).to_string(index=False))
rep.add_table('deprivation gradient on slope adjusted distance', GRAD, max_rows=20)

base = GRAD[(GRAD.variant == 'flat')].set_index('destination set')['coefficient']
print(f"\nbaseline refitted here: {ANCHORS[0]} {base[ANCHORS[0]]:+.4f}, "
      f"{ANCHORS[1]} {base[ANCHORS[1]]:+.4f}")
for k, lab in VARIANTS[1:]:
    g = GRAD[GRAD.variant == k].set_index('destination set')['coefficient']
    print(f'{lab}: {ANCHORS[0]} {g[ANCHORS[0]]:+.4f} '
          f'({100 * (g[ANCHORS[0]] / base[ANCHORS[0]] - 1):+.1f} per cent of the baseline), '
          f'{ANCHORS[1]} {g[ANCHORS[1]]:+.4f} '
          f'({100 * (g[ANCHORS[1]] / base[ANCHORS[1]] - 1):+.1f} per cent)')

## 8. Write results

The block file, the area file and the fitted gradients are written to the
results folder, and the report is saved and offered for download.

In [ ]:
keep = (['CVEGEO_MZA', 'slug', 'POBTOT']
        + [f'minutes_flat_{s}' for s in ANCHORS]
        + [f'minutes_to_{s}' for s in ANCHORS]
        + [f'minutes_back_{s}' for s in ANCHORS]
        + [f'slope_m_to_{s}' for s in ANCHORS]
        + [f'slope_m_back_{s}' for s in ANCHORS]
        + [f'slope_m_round_{s}' for s in ANCHORS])
out = os.path.join(RESULTS, '13_block_slope.parquet')
A[keep].to_parquet(out, index=False)
print('written:', out, f'({os.path.getsize(out) / 1e6:.1f} MB)')
outa = os.path.join(RESULTS, '13_area_slope.parquet')
AREA.to_parquet(outa, index=False)
print('written:', outa)
outg = os.path.join(RESULTS, '13_slope_gradient.parquet')
GRAD.to_parquet(outg, index=False)
print('written:', outg)

g_to = GRAD[GRAD.variant == 'to'].set_index('destination set')
rep.note(f'{len(A):,} inhabited blocks timed over the terrain of {len(AREA)} areas; '
         f'{100 * STRATA.loc[0, "share_length_grade_over_5pct"]:.1f} per cent of network '
         f'length steeper than five per cent; median walk to a supermarket or club '
         f'{STRATA.loc[0, "flat_m_supermarket_club"]:,} m flat against '
         f'{STRATA.loc[0, "to_m_supermarket_club"]:,} m slope adjusted; gradient '
         f'{g_to.loc[ANCHORS[0], "coefficient"]:+.4f} ({g_to.loc[ANCHORS[0], "standard error"]:.4f}) '
         f'for supermarkets and {g_to.loc[ANCHORS[1], "coefficient"]:+.4f} '
         f'({g_to.loc[ANCHORS[1], "standard error"]:.4f}) for fresh sellers on the walk '
         f'to the outlet, against {base[ANCHORS[0]]:+.4f} and {base[ANCHORS[1]]:+.4f} flat')
rep.save()